<a href="https://colab.research.google.com/github/Kabir-Khan7/AI-Agent/blob/main/Rag_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Downloading Ollama Binary File

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,971 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama serve &
!ollama pull qwen2.5:7b

Error: listen tcp 127.0.0.1:11434: bind: address already in use



## Installing Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-ollama chromadb openpyxl pandas

## Check the Ollama Responding

In [ ]:
import os
import subprocess
import time
import requests

# 1. Kill any existing ollama processes to start fresh
!pkill ollama

# 2. Set Environment Variables so Ollama knows exactly where to listen
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

# 3. Start the server in the background
print("Starting Ollama server...")
subprocess.Popen(["/usr/local/bin/ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)

# 4. WAIT and VERIFY (Don't just sleep, actually check if it's awake)
print("Waiting for server to respond...", end="")
for i in range(20):
    try:
        res = requests.get("http://127.0.0.1:11434")
        if res.status_code == 200:
            print("\n✅ Server is UP and Running!")
            break
    except:
        print(".", end="")
        time.sleep(2)
else:
    print("\n❌ Server failed to start after 40 seconds.")

Starting Ollama server...
Waiting for server to respond....
✅ Server is UP and Running!


## Clearing space of Chroma DB (OPITIONAL)

In [ ]:
# import shutil
# import os

# # 1. Force remove the old database folders
# paths_to_clear = ["./chroma_db_tiering", "./chroma_db_new"]
# for path in paths_to_clear:
#     if os.path.exists(path):
#         shutil.rmtree(path)
#         print(f"Cleared {path}")

# # 2. Restart the Python kernel (Optional but recommended)
# # After running this, run your Server Start cell and then the Main Code cell.

# RAG System Code

In [ ]:
import os
import pandas as pd
import requests
import shutil
import time
import uuid
from datetime import datetime
from typing import List

# LangChain core
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# Ollama integration
from langchain_ollama import OllamaLLM, OllamaEmbeddings

# Vector store
from langchain_community.vectorstores import Chroma

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
OLLAMA_BASE_URL  = "http://127.0.0.1:11434"
OLLAMA_MODEL     = "qwen2.5:7b"
EMBED_MODEL      = "nomic-embed-text"
EXCEL_FILE       = "product_details.xlsx"

# We create a UNIQUE folder name every time the script runs to avoid "Read-Only" locks
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
CHROMA_DIR = f"/content/chroma_db_{RUN_ID}"

CHUNK_SIZE       = 1000
CHUNK_OVERLAP    = 100

# ─────────────────────────────────────────────
# STEP 1 — Load & Verify Excel
# ─────────────────────────────────────────────
def load_excel_as_documents(file_path: str) -> List[Document]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' not found! Upload it to the Colab sidebar.")

    df = pd.read_excel(file_path)
    print(f"\n✅ SUCCESS: Found {file_path}")
    print(f"📊 Preview of your LATEST data (First 3 rows):")
    print(df.head(3))
    print("-" * 30)

    documents: List[Document] = []
    for _, row in df.iterrows():
        text_parts = [f"{col}: {row[col]}" for col in df.columns if pd.notna(row[col])]
        page_content = "\n".join(text_parts)
        metadata = {str(k): str(v) for k, v in row.items() if pd.notna(v)}
        documents.append(Document(page_content=page_content, metadata=metadata))
    return documents

# ─────────────────────────────────────────────
# STEP 2 — Split
# ─────────────────────────────────────────────
def split_documents(documents: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", " ", ""],
    )
    return splitter.split_documents(documents)

# ─────────────────────────────────────────────
# STEP 3 — Create Vector Store
# ─────────────────────────────────────────────
def build_vector_store(chunks: List[Document]) -> Chroma:
    # Ensure Ollama is running
    try:
        requests.get(OLLAMA_BASE_URL)
    except:
        raise ConnectionError("Ollama server not found. Please run the server cell!")

    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

    print(f"📦 Creating a fresh database at: {CHROMA_DIR}")
    print(f"🔄 Embedding {len(chunks)} chunks... (This may take a minute)")

    # Initialize Chroma with the unique directory
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DIR
    )
    return db

# ─────────────────────────────────────────────
# STEP 4 — Build RAG Chain
# ─────────────────────────────────────────────
def build_rag_chain(vector_store: Chroma) -> RetrievalQA:
    llm = OllamaLLM(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.1)
    retriever = vector_store.as_retriever(search_kwargs={"k": 15})

    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="""You are an expert sales data analyst. Use ONLY the data provided.
        Evaluate and assign categories (Bronze, Silver, Gold) based on the figures.

        DATA:
        {context}

        INSTRUCTION: {question}

        ANSWER:"""
    )

    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_template},
    )

# ─────────────────────────────────────────────
# RUN SYSTEM
# ─────────────────────────────────────────────
def run():
    print("=" * 60)
    print("  RAG SYSTEM: FORCING FRESH DATA LOAD")
    print("=" * 60)

    try:
        # Load and Preview
        docs = load_excel_as_documents(EXCEL_FILE)

        # Build
        chunks = split_documents(docs)
        vector_db = build_vector_store(chunks)
        rag_chain = build_rag_chain(vector_db)

        print(f"\n✅ SYSTEM ONLINE!")
        print(f"📂 New database created at: {CHROMA_DIR}")

        while True:
            query = input("\n🔍 Enter instruction: ").strip()
            if query.lower() in ['q', 'quit', 'exit']:
                break
            if not query: continue

            print("⏳ LLM is analyzing...")
            res = rag_chain.invoke({"query": query})
            print(f"\n💡 Answer:\n{res['result']}")

    except Exception as e:
        print(f"\n❌ FAILED: {e}")
        print("\n💡 TIP: If you see a 'readonly' error again, go to the top menu:")
        print("   Runtime -> Restart Session, then run the cells again.")

if __name__ == "__main__":
    run()

  RAG SYSTEM: FORCING FRESH DATA LOAD

✅ SUCCESS: Found product_details.xlsx
📊 Preview of your LATEST data (First 3 rows):
        Barcode          ABO SKU  \
0  831181032529     CABTEQ001189   
1  833334664651     CABTAM001199   
2  812659322590  ELLC/SKU/001386   

                                        Product Name  Febuary Sales  \
0  Electro Vouge - Vouge Bluetooth Headset 2 in1 ...             50   
1  ElectroVouge - Vouge Bluetooth Headset 2in1 - ...            100   
2  Electro Paw II - Dog Style Portal Bluetooth Sp...            120   

   March Sales  
0         1000  
1         2550  
2         3600  
------------------------------
📦 Creating a fresh database at: /content/chroma_db_20260421_201817
🔄 Embedding 100 chunks... (This may take a minute)

✅ SYSTEM ONLINE!
📂 New database created at: /content/chroma_db_20260421_201817

🔍 Enter instruction: Can you analysze all product and assigned tiers bronze, silver and gold
⏳ LLM is analyzing...

💡 Answer:
Certainly! To categoriz